In [12]:
from typing import Any, Callable, Optional, Union

from pprint import pprint
from datetime import datetime
from pathlib import Path
import os

from torch.utils.tensorboard import SummaryWriter
import pandas as pd
import pytz
import numpy as np
import safetensors.torch as safetensors
import tqdm.notebook as tqdm
import torch
import torch.utils.data as torchdata
import torch.nn as nn
import torchmetrics
import yaml

from flatiron.core.dataset import Dataset
from flatiron.core.types import Compiled, Filepath, Getter

import torch._dynamo
torch._dynamo.config.suppress_errors = True

Filepath = Union[str, Path]

In [13]:
# CALLBACKS
def get_tensorboard_project(
    project, root='/mnt/storage', timezone='America/Detroit', extension='safetensors'
):
    # type: (Filepath, Filepath, str, str) -> dict[str, str]
    '''
    Creates directory structure for Tensorboard project.

    Args:
        project (str): Name of project.
        root (str or Path): Tensorboard parent directory. Default: /mnt/storage
        timezone (str, optional): Timezone. Default: America/Detroit.
        extension (str, optional): File extension.
            Options: [pth, safetensors]. Default: keras.

    Raises:
        EnforceError: If extension is not keras or safetensors.

    Returns:
        dict: Project details.
    '''
    msg = 'Extension must be pth or safetensors. Given value: {extension}.'
    assert extension in ['pth', 'safetensors'], msg
    # --------------------------------------------------------------------------

    # create timestamp
    timestamp = datetime \
        .now(tz=pytz.timezone(timezone)) \
        .strftime('d-%Y-%m-%d_t-%H-%M-%S')

    # create directories
    root_dir = Path(root, project, 'tensorboard').as_posix()
    log_dir = Path(root_dir, timestamp).as_posix()
    model_dir = Path(log_dir, 'models').as_posix()
    os.makedirs(root_dir, exist_ok=True)
    os.makedirs(model_dir, exist_ok=True)

    # checkpoint pattern
    epoch = '{epoch:03d}'
    target = f'p-{project}_{timestamp}_e-{epoch}.{extension}'
    target = Path(model_dir, target).as_posix()

    output = dict(
        root_dir=root_dir,
        log_dir=log_dir,
        model_dir=model_dir,
        checkpoint_pattern=target,
    )
    return output


class ModelCheckpoint:
    '''
    Class for saving PyTorch models.
    '''
    def __init__(self, filepath, save_freq='epoch', **kwargs):
        # type: (Filepath, str, Any) -> None
        '''
        Constructs ModelCheckpoint instance.

        Args:
            filepath (str or Path): Filepath pattern.
            save_freq (str, optional): Save frequency. Default: epoch.
        '''
        self._filepath = Path(filepath).as_posix()
        self.save_freq = save_freq

    def save(self, model, epoch):
        # type: (torch.nn.Module, int) -> None
        '''
        Save PyTorch model.

        Args:
            model (torch.nn.Module): Model to be saved.
            epoch (int): Current epoch.
        '''
        filepath = self._filepath.format(epoch=epoch)
        safetensors.save_model(model, filepath)


Callbacks = dict[str, SummaryWriter | ModelCheckpoint]


def get_callbacks(log_directory, checkpoint_pattern, checkpoint_params={}):
    # type: (Filepath, str, dict) -> Callbacks
    '''
    Create a list of callbacks for Tensoflow model.

    Args:
        log_directory (str or Path): Tensorboard project log directory.
        checkpoint_pattern (str): Filepath pattern for checkpoint callback.
        checkpoint_params (dict, optional): Params to be passed to checkpoint
            callback. Default: {}.

    Raises:
        EnforceError: If log directory does not exist.
        EnforeError: If checkpoint pattern does not contain '{epoch}'.

    Returns:
        list: Tensorboard and ModelCheckpoint callbacks.
    '''
    return dict(
        tensorboard=SummaryWriter(log_dir=log_directory),
        checkpoint=ModelCheckpoint(checkpoint_pattern, **checkpoint_params),
    )

In [14]:
# TORCHDATASET CLASS
class TorchDataset(Dataset, torchdata.Dataset):
    '''
    Class for inheriting torch Dataset into flatiron Dataset.
    '''
    @staticmethod
    def monkey_patch(dataset, channels_first=True):
        # type: (Dataset, bool) -> TorchDataset
        '''
        Construct and monkey patch a new TorchDataset instance from a given
        Dataset.
        Pytorch expects images in with the shape (C, H , W) per default.

        Args:
            dataset (Dataset): Dataset.
            channels_first (bool, optional): Will convert any matrix of shape
                (H, W, C)  into (C, H, W). Default: True.

        Returns:
            TorchDataset: TorchDataset instance.
        '''
        this = TorchDataset(dataset.info)
        this._info = dataset._info.copy()
        this._info['frame'] = this._info.index
        this.data = dataset.data
        this.labels = dataset.labels
        this.label_axis = dataset.label_axis
        this._ext_regex = dataset._ext_regex
        this._calc_file_size = dataset._calc_file_size
        this._sample_gb = dataset._sample_gb
        this._channels_first = channels_first  # type: ignore
        return this

    def __getitem__(self, frame):
        # type: (int) -> list[torch.Tensor]
        '''
        Get tensor data by frame.

        Returns:
            lis[torch.Tensor]: List of Tensors.
        '''
        items = self.get_arrays(frame)

        # pytorch warns about arrays not being writable, this fixes that
        items = [x.copy() for x in items]

        # pytorch expects (C, H, W) because it sucks
        if self._channels_first:  # type: ignore
            arrays = items
            items = []
            for item in arrays:
                if item.ndim == 3:
                    item = np.transpose(item, (2, 0, 1))
                items.append(item)

        output = list(map(torch.from_numpy, items))
        return output

In [15]:
# TRAIN FUNCS
def _execute_epoch(
    epoch,             # type: int
    model,             # type: torch.nn.Module
    data_loader,       # type: torch.utils.data.DataLoader
    optimizer,         # type: torch.optim.Optimizer
    loss_func,         # type: torch.nn.Module
    device,            # type: torch.device
    metrics_funcs=[],  # type: list[Callable]
    writer=None,       # type: Optional[SummaryWriter]
    checkpoint=None,   # type: Optional[ModelCheckpoint]
    mode='train',      # type: str
):
    # type: (...) -> None
    '''
    Execute train or test epoch on given torch model.

    Args:
        epoch (int): Current epoch.
        model (torch.nn.Module): Torch model.
        data_loader (torch.utils.data.DataLoader): Torch data loader.
        optimizer (torch.optim.Optimizer): Torch optimizer.
        loss_func (torch.nn.Module): Torch loss function.
        metrics_funcs (list[Callable], optional): List of torch metrics.
            Default: [].
        writer (SummaryWriter, optional): Tensorboard writer. Default: None.
        checkpoint (ModelCheckpoint, optional): Model saver. Default: None.
        device (torch.device): Torch device.
        mode (str, optional): Mode to execute. Options: [train, test].
            Default: train.
    '''
    if mode == 'train':
        context = torch.enable_grad  # type: Any
        model.train()
    elif mode == 'test':
        context = torch.inference_mode
        model.eval()
    else:
        raise ValueError(f'Invalid mode: {mode}.')

    # checkpoint mode
    checkpoint_mode = checkpoint is not None and checkpoint.save_freq == 'batch'

    metrics = []
    epoch_size = len(data_loader)
    with context():
        for i, batch in enumerate(data_loader):
            # get x and y
            if len(batch) == 2:
                x, y = batch
                x = x.to(device)
                y = y.to(device)
            else:
                x = batch
                x = x.to(device)
                y = x

            y_pred = model(x)
            loss = loss_func(y_pred, y)

            # train model on batch
            if mode == 'train':
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            # gather batch metrics
            batch_metrics = dict(loss=loss)
            for metric in metrics_funcs:
                batch_metrics[metric.__class__.__name__] = metric(y_pred, y)
            metrics.append(batch_metrics)

            # write batch metrics
            if writer is not None:
                batch_index = epoch * epoch_size + i
                for key, val in batch_metrics.items():
                    writer.add_scalar(f'{mode}_batch_{key}', val, batch_index)

            # save model
            if checkpoint_mode:
                checkpoint.save(model, epoch)  # type: ignore

    # write mean epoch metrics
    if writer is not None:
        epoch_metrics = pd.DataFrame(metrics) \
            .map(lambda x: x.cpu().detach().numpy().mean()) \
            .rename(lambda x: f'{mode}_epoch_{x}', axis=1) \
            .mean() \
            .to_dict()

        for key, val in epoch_metrics.items():
            writer.add_scalar(f'{mode}_epoch_{key}', val, epoch * epoch_size)


def train(
    device,      # type: str
    model,       # type: torch.nn.Module
    optimizer,   # type: torch.optim.Optimizer
    loss,        # type: torch.nn.Module
    metrics,     # type: list[torch.nn.Module]
    callbacks,   # type: Callbacks
    train_data,  # type: Dataset
    test_data,   # type: Dataset
    params,      # type: dict
):
    # type: (...) -> None
    '''
    Train Torch model.

    Args:
        device (str): Device to compile to.
        model (torch.nn.Module): Model to be compiled.
        optimizer (dict): Optimizer config for compilation.
        loss (str): Loss to be compiled.
        metrics (list[str]): Metrics function to be compiled.
        callbacks (dict): Dict of callbacks.
        train_data (Dataset): Training dataset.
        test_data (Dataset): Test dataset.
        params (dict): Training params.
    '''
    checkpoint = callbacks['checkpoint']  # type: Any
    writer = callbacks['tensorboard']
    batch_size = params['batch_size']

    device = torch.device(device)
    torch.manual_seed(params['seed'])
    model = model.to(device)

    train_loader = torchdata.DataLoader(
        TorchDataset.monkey_patch(train_data), batch_size=batch_size
    )  # type: torchdata.DataLoader
    test_loader = torchdata.DataLoader(
        TorchDataset.monkey_patch(test_data), batch_size=batch_size
    )  # type: torchdata.DataLoader

    kwargs = dict(
        model=model,
        optimizer=optimizer,
        loss_func=loss.to(device),
        device=device,
        metrics_funcs=[x.to(device) for x in metrics],
        writer=writer,
    )
    for i in tqdm.trange(params['epochs']):
        _execute_epoch(
            epoch=i, mode='train', data_loader=train_loader,
            checkpoint=checkpoint, **kwargs
        )
        _execute_epoch(epoch=i, mode='test', data_loader=test_loader, **kwargs)
        if checkpoint.save_freq == 'epoch':
            checkpoint.save(model, i)

In [16]:
# DATA
data_kwargs = dict(
    directory='/mnt/storage/projects/unet001/dset002/p-unet001_s-dset002_d-glom_v001',
    label_axis=-1,
    labels=['a'],
)
data = Dataset.read_directory(**data_kwargs)
train_data, test_data = data.train_test_split()
print('DATA')
pprint(data_kwargs)

DATA
{'directory': '/mnt/storage/projects/unet001/dset002/p-unet001_s-dset002_d-glom_v001',
 'label_axis': -1,
 'labels': ['a']}


In [17]:
# MODEL ARCHITECTURE
class Model(nn.Module):
    def __init__(self, input_channels, output_channels):
        super().__init__()
        self.layer_stack = nn.Sequential(
            nn.Conv2d(
                in_channels=input_channels, out_channels=output_channels,
                kernel_size=(3, 3), dtype=torch.float16, padding=1
            ),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.layer_stack(x)

In [18]:
# MODEL
model_kwargs = dict(
    input_channels=3,
    output_channels=1,
)
model = Model(**model_kwargs)
print('MODEL')
pprint(model_kwargs)

MODEL
{'input_channels': 3, 'output_channels': 1}


In [19]:
# CALLBACKS
tb = get_tensorboard_project(
    project='unet001',
    root='/mnt/storage/projects',
)
print('TENSORBOARD')
pprint(tb)

callback_kwargs = dict(
    log_directory=tb['log_dir'],
    checkpoint_pattern=tb['checkpoint_pattern'],
    checkpoint_params=dict(save_freq='epoch'),
)
callbacks = get_callbacks(**callback_kwargs)
print()
print('CALLBACKS')
pprint(callback_kwargs)

# TRAIN KWARGS
train_kwargs = dict(
    device='cuda',
    model=torch.compile(model),
    optimizer=torch.optim.SGD(
        model.parameters(),
        lr=0.001,
    ),
    loss=nn.modules.loss.MSELoss(),
    metrics=[
        torchmetrics.MeanMetric(),
    ],
    callbacks=callbacks,
    train_data=train_data,
    test_data=test_data,
    params=dict(
        epochs=10,
        seed=42,
        batch_size=32,
    )
)
print()
print('TRAIN')
pprint(train_kwargs)

# TRAIN
print()
train(**train_kwargs)

TENSORBOARD
{'checkpoint_pattern': '/mnt/storage/projects/unet001/tensorboard/d-2025-02-26_t-13-14-15/models/p-unet001_d-2025-02-26_t-13-14-15_e-{epoch:03d}.safetensors',
 'log_dir': '/mnt/storage/projects/unet001/tensorboard/d-2025-02-26_t-13-14-15',
 'model_dir': '/mnt/storage/projects/unet001/tensorboard/d-2025-02-26_t-13-14-15/models',
 'root_dir': '/mnt/storage/projects/unet001/tensorboard'}

CALLBACKS
{'checkpoint_params': {'save_freq': 'epoch'},
 'checkpoint_pattern': '/mnt/storage/projects/unet001/tensorboard/d-2025-02-26_t-13-14-15/models/p-unet001_d-2025-02-26_t-13-14-15_e-{epoch:03d}.safetensors',
 'log_directory': '/mnt/storage/projects/unet001/tensorboard/d-2025-02-26_t-13-14-15'}

TRAIN
{'callbacks': {'checkpoint': <__main__.ModelCheckpoint object at 0x7f2144fa0430>,
               'tensorboard': <torch.utils.tensorboard.writer.SummaryWriter object at 0x7f2144fa0250>},
 'device': 'cuda',
 'loss': MSELoss(),
 'metrics': [MeanMetric()],
 'model': OptimizedModule(
  (_orig_m

  0%|          | 0/10 [00:00<?, ?it/s]

W0226 18:14:16.213000 688887 torch/_inductor/utils.py:1137] [1/0] Not enough SMs to use max_autotune_gemm mode
W0226 18:14:16.246000 688887 torch/_dynamo/convert_frame.py:1233] WON'T CONVERT forward /tmp/ipykernel_688887/2959099092.py line 13 
W0226 18:14:16.246000 688887 torch/_dynamo/convert_frame.py:1233] due to: 
W0226 18:14:16.246000 688887 torch/_dynamo/convert_frame.py:1233] Traceback (most recent call last):
W0226 18:14:16.246000 688887 torch/_dynamo/convert_frame.py:1233]   File "/home/ubuntu/pdm/envs/pdm-kVbOHlCT-dev-3.10/lib/python3.10/site-packages/torch/_dynamo/convert_frame.py", line 1164, in __call__
W0226 18:14:16.246000 688887 torch/_dynamo/convert_frame.py:1233]     result = self._inner_convert(
W0226 18:14:16.246000 688887 torch/_dynamo/convert_frame.py:1233]   File "/home/ubuntu/pdm/envs/pdm-kVbOHlCT-dev-3.10/lib/python3.10/site-packages/torch/_dynamo/convert_frame.py", line 547, in __call__
W0226 18:14:16.246000 688887 torch/_dynamo/convert_frame.py:1233]     retur

In [22]:
!exa --tree /mnt/storage/projects/unet001/tensorboard/d-2025*

/mnt/storage/projects/unet001/tensorboard/d-2025-02-25_t-13-44-28
├── events.out.tfevents.1740509068.5abe6464f7f1.258531.1
└── models
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-000.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-001.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-002.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-003.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-004.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-005.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-006.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-007.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-008.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-009.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-010.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-011.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-012.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-013.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28